In [1]:
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input,Dropout

import keras_tuner as kt

In [3]:
df = pd.read_csv(r"C:\Users\divya\Projects\EDA\diabetes.csv")

In [4]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

In [6]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [7]:
X = scaler.fit_transform(X)

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [9]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense

In [10]:
model = Sequential()
model.add(Dense(32,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

C:\Users\divya\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
model.fit(X_train,y_train, batch_size=32, epochs=10,validation_data=(X_test,y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.5912 - loss: 0.6707 - val_accuracy: 0.6753 - val_loss: 0.6268
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7003 - loss: 0.6198 - val_accuracy: 0.7403 - val_loss: 0.5926
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7329 - loss: 0.5822 - val_accuracy: 0.7532 - val_loss: 0.5676
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7459 - loss: 0.5533 - val_accuracy: 0.7662 - val_loss: 0.5479
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7557 - loss: 0.5301 - val_accuracy: 0.7532 - val_loss: 0.5335
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7508 - loss: 0.5128 - val_accuracy: 0.7792 - val_loss: 0.5231
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7573 - loss: 0.4980 - val_accuracy: 0.7662 - val_loss: 0.5160
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7671 - loss: 0.4872 - val_accuracy: 0.7662 - val_loss

In [11]:
# 1. how to select appropriate optimizer
# 2. No, of nodes in a layer
# 3. how to select no. of layers
# 4. All in all one model

# 1. how to select appropriate optimizer

In [12]:
def build_model(hp):

    model = Sequential()

    model.add(Input(shape=(8,)))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    optimizer = hp.Choice(
        "optimizer",
        ["adam", "sgd", "rmsprop", "adadelta"]
    )

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [13]:
model = build_model(kt.HyperParameters())

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                      │ (None, 32)                  │             288 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
tuner = kt.RandomSearch(
    hypermodel=build_model,
    objective="val_accuracy",
    max_trials=5,
    overwrite=True,
    directory="my_dir",
    project_name="optimizer_search"
)

In [15]:
tuner.search_space_summary()

Search space summary
Default search space size: 1
optimizer (Choice)
{'default': 'adam', 'conditions': [], 'values': ['adam', 'sgd', 'rmsprop', 'adadelta'], 'ordered': False}


In [16]:
keras.config.disable_traceback_filtering()

In [17]:
tuner.search(
    X_train,
    y_train,
    epochs=5,
    validation_data=(X_test, y_test),
    verbose=2
)

Trial 4 Complete [00h 00m 02s]
val_accuracy: 0.7662337422370911

Best val_accuracy So Far: 0.7662337422370911
Total elapsed time: 00h 00m 09s


In [18]:
print(tuner.oracle.trials)

{'0': <keras_tuner.src.engine.trial.Trial object at 0x00000171E6F16270>, '1': <keras_tuner.src.engine.trial.Trial object at 0x00000171E9916210>, '2': <keras_tuner.src.engine.trial.Trial object at 0x00000171E9916710>, '3': <keras_tuner.src.engine.trial.Trial object at 0x00000171E9AFF490>}


In [19]:
tuner.results_summary()

Results summary
Results in my_dir\optimizer_search
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 3 summary
Hyperparameters:
optimizer: adam
Score: 0.7662337422370911

Trial 0 summary
Hyperparameters:
optimizer: rmsprop
Score: 0.6948052048683167

Trial 1 summary
Hyperparameters:
optimizer: sgd
Score: 0.6428571343421936

Trial 2 summary
Hyperparameters:
optimizer: adadelta
Score: 0.3571428656578064


In [20]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print(best_hp.values)

{'optimizer': 'adam'}


In [21]:
model = tuner.get_best_models(num_models=1)[0]

C:\Users\divya\anaconda3\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [22]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 32)                  │             288 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7524 - loss: 0.5589 - val_accuracy: 0.7922 - val_loss: 0.5525
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7622 - loss: 0.5307 - val_accuracy: 0.7922 - val_loss: 0.5350
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7638 - loss: 0.5112 - val_accuracy: 0.7857 - val_loss: 0.5214
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7720 - loss: 0.4973 - val_accuracy: 0.7922 - val_loss: 0.5115
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7752 - loss: 0.4862 - val_accuracy: 0.7857 - val_loss: 0.5071
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7752 - loss: 0.4786 - val_accuracy: 0.7792 - val_loss: 0.5033
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7785 - loss: 0.4712 - val_accuracy: 0.7792 - val_loss: 0.4996
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7785 - loss: 0.4654 - val_accuracy: 0.77

# 2. No. of nodes in a layer

In [24]:
def build_model(hp):

    model = Sequential()

    units = hp.Int('units',8,128,step=8)

    model.add(Dense(units=units, activation='relu',input_dim=8))
    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['accuracy' ])

    return model

In [25]:
tuner = kt.RandomSearch(
    hypermodel=build_model,
    objective="val_accuracy",
    max_trials=5,
    overwrite=True,
    directory="my_dir",
    project_name="optimizer_search"
)

C:\Users\divya\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [26]:
tuner.search(
    X_train,
    y_train,
    epochs=5,
    validation_data=(X_test, y_test),
    verbose=2
)

Trial 5 Complete [00h 00m 02s]
val_accuracy: 0.7532467246055603

Best val_accuracy So Far: 0.7662337422370911
Total elapsed time: 00h 00m 12s


In [27]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]

print(best_hp.values)

{'units': 96}


In [28]:
model = tuner.get_best_models(num_models=1)[0]  # it will give the best model

C:\Users\divya\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\divya\anaconda3\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [29]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7573 - loss: 0.4872 - val_accuracy: 0.7792 - val_loss: 0.4995
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7704 - loss: 0.4698 - val_accuracy: 0.7857 - val_loss: 0.4942
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7720 - loss: 0.4614 - val_accuracy: 0.7857 - val_loss: 0.4949
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7834 - loss: 0.4548 - val_accuracy: 0.7727 - val_loss: 0.4984
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7801 - loss: 0.4496 - val_accuracy: 0.7662 - val_loss: 0.4993
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7850 - loss: 0.4454 - val_accuracy: 0.7792 - val_loss: 0.4999
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7818 - loss: 0.4425 - val_accuracy: 0.7597 - val_loss: 0.5017
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7866 - loss: 0.4395 - val_accuracy: 0.76

## How to select no. of layers?

In [30]:
def build_model(hp):

    model = Sequential()

    model.add(Dense(72,activation='relu',input_dim=8))

    for i in range(hp.Int('num_layers',min_value=1,max_value=10)):

        model.add(Dense(72,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))

    model.compile(optimizer='rmsprop', loss='binary_crossentropy',metrics=['accuracy'])

    return model

In [31]:
tuner = kt.RandomSearch(build_model,
objective='val_accuracy',
max_trials=3,
directory='mydir',
project_name='num_layers')

C:\Users\divya\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [32]:
tuner.search(X_train,y_train, epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 04s]
val_accuracy: 0.7727272510528564

Best val_accuracy So Far: 0.7857142686843872
Total elapsed time: 00h 00m 10s


In [33]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]  # num_trials 1 means give me the best 1 trail

print(best_hp.values)

{'num_layers': 1}


## 4. All in all one model

In [34]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

def build_model(hp):

    model = Sequential()

    counter = 0

    for i in range(hp.Int('num_layers', min_value=1, max_value=10)):

        if counter == 0:
            
            model.add(Dense(
                units=hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
                activation=hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid']),
                input_dim=8
            ))
            model.add(Dropout(hp.Choice('dropout'+str(i), values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))

        else:

            model.add(Dense(
                units=hp.Int('units' + str(i), min_value=8, max_value=128, step=8),
                activation=hp.Choice('activation' + str(i), values=['relu', 'tanh', 'sigmoid'])
            ))
            model.add(Dropout(hp.Choice('dropout'+str(i), values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))

        counter += 1

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=hp.Choice('optimizer', values=['rmsprop', 'adam', 'sgd', 'nadam', 'adadelta']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

"""This model allows Keras Tuner to search for:

Number of hidden layers

hp.Int('num_layers', min_value=1, max_value=10)

→ Between 1 and 10 hidden layers.

Number of neurons in each hidden layer

hp.Int('units0', ...)
hp.Int('units1', ...)
hp.Int('units2', ...)

→ Each layer can have 8, 16, 24, ..., 128 neurons.

Activation function for each hidden layer

hp.Choice(
    'activation0',
    ['relu', 'tanh', 'sigmoid']
)

→ Each layer can independently choose ReLU, Tanh, or Sigmoid.

Optimizer

hp.Choice(
    'optimizer',
    ['rmsprop', 'adam', 'sgd', 'nadam', 'adadelta']
)"""

"This model allows Keras Tuner to search for:\n\nNumber of hidden layers\n\nhp.Int('num_layers', min_value=1, max_value=10)\n\n→ Between 1 and 10 hidden layers.\n\nNumber of neurons in each hidden layer\n\nhp.Int('units0', ...)\nhp.Int('units1', ...)\nhp.Int('units2', ...)\n\n→ Each layer can have 8, 16, 24, ..., 128 neurons.\n\nActivation function for each hidden layer\n\nhp.Choice(\n    'activation0',\n    ['relu', 'tanh', 'sigmoid']\n)\n\n→ Each layer can independently choose ReLU, Tanh, or Sigmoid.\n\nOptimizer\n\nhp.Choice(\n    'optimizer',\n    ['rmsprop', 'adam', 'sgd', 'nadam', 'adadelta']\n)"

In [35]:
tuner = kt.RandomSearch(
    hypermodel=build_model,
    objective="val_accuracy",
    max_trials=3,
    overwrite=True,
    directory="my_dir",
    project_name="optimizer_search"
)

C:\Users\divya\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [36]:
tuner.search(X_train,y_train, epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.649350643157959
Total elapsed time: 00h 00m 10s


In [37]:
tuner.get_best_hyperparameters(num_trials=1)[0].values

{'num_layers': 5,
 'units0': 72,
 'activation0': 'tanh',
 'dropout0': 0.4,
 'optimizer': 'sgd',
 'units1': 8,
 'activation1': 'relu',
 'dropout1': 0.1,
 'units2': 8,
 'activation2': 'relu',
 'dropout2': 0.1,
 'units3': 8,
 'activation3': 'relu',
 'dropout3': 0.1,
 'units4': 8,
 'activation4': 'relu',
 'dropout4': 0.1}

In [38]:
model = tuner.get_best_models(num_models=1)[0]  # it will give the best model

C:\Users\divya\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [39]:
model.fit(X_train,y_train,batch_size=32,epochs=200,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6450 - loss: 0.6755 - val_accuracy: 0.6429 - val_loss: 0.6701
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6645 - loss: 0.6641 - val_accuracy: 0.6429 - val_loss: 0.6655
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6629 - loss: 0.6676 - val_accuracy: 0.6429 - val_loss: 0.6613
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6417 - loss: 0.6623 - val_accuracy: 0.6429 - val_loss: 0.6576
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6515 - loss: 0.6614 - val_accuracy: 0.6429 - val_loss: 0.6548
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6580 - loss: 0.6535 - val_accuracy: 0.6429 - val_loss: 0.6521
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6547 - loss: 0.6523 - val_accuracy: 0.6429 - val_loss: 0.6494
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6515 - loss: 0.6481 - val_accuracy: 0.64

In [40]:
# 1. Build a HyperModel
#           │
#           ▼
# 2. Define the Search Space
#           │
#           ▼
# 3. Create a Tuner
#           │
#           ▼
# 4. Search Best Hyperparameters
#           │
#           ▼
# 5. Get Best Hyperparameters
#           │
#           ▼
# 6. Build Final Model
#           │
#           ▼
# 7. Train Final Model
#           │
#           ▼
# 8. Evaluate Model